# LLM capability-floor smoke (M4)

**What this does:** makes a small open model (Qwen2.5-3B) *play* the beer game, with no training. This is the free "can the model even play coherently?" check that PROJECT_SPEC calls for **before** any paid GRPO run.

**How to use:** Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> **Run all**. First run takes a few minutes (it downloads the model). Total cost: **$0** on Colab free tier.

**What to look for:** in the last cell's output, does the model keep inventory and cost in a sane range (not ordering 0 forever or slamming the cap), and is `parse_fail_rate` near 0? If yes, the capability floor passes and GRPO is worth doing. If the model plays incoherently, the spec says move up a model size before spending anything.

In [ ]:
#@title 1) Install Ollama + start the model server
# Ollama is a tiny program that runs an open LLM locally and answers requests.
# Our decoder talks to it over http://127.0.0.1:11434 (nothing to configure).
import subprocess, time, os
subprocess.run("curl -fsSL https://ollama.ai/install.sh | sh", shell=True, check=True)
# start the server in the background
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
server = subprocess.Popen(["ollama", "serve"])
time.sleep(5)
print("ollama server started")

In [ ]:
#@title 2) Download the model (Qwen2.5-3B)
MODEL = "qwen2.5:3b"  #@param {type:"string"}
import subprocess
subprocess.run(["ollama", "pull", MODEL], check=True)
print("pulled", MODEL)

In [ ]:
#@title 3) Clone the repo + install the environment
REPO_URL = "https://github.com/ConstantinVictorBeatErtel/beer_distribution_RL.git"  #@param {type:"string"}
BRANCH = "claude/project-status-summary-zy3kht"  #@param {type:"string"}
import pathlib, subprocess, sys
repo = pathlib.Path("/content/beer_distribution_RL")
if not repo.exists():
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(repo)])
else:
    subprocess.check_call(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", BRANCH])
    subprocess.check_call(["git", "-C", str(repo), "checkout", BRANCH])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=str(repo))
print("repo ready on branch", BRANCH)

In [ ]:
#@title 4a) Sanity check with NO model (demand-matching baseline)
# Proves the wiring works and gives the cost the LLM has to beat/match.
import subprocess, sys
subprocess.check_call([
    sys.executable, "scripts/run_llm_episode.py",
    "--backend", "heuristic", "--cell", "classic", "--horizon", "52",
], cwd="/content/beer_distribution_RL")

In [ ]:
#@title 4b) The real capability floor: let Qwen play (classic cell)
# ~1 order per role per week x 52 weeks x 4 roles = ~200 model calls. A few minutes on T4.
import subprocess, sys
MODEL = "qwen2.5:3b"
subprocess.check_call([
    sys.executable, "scripts/run_llm_episode.py",
    "--backend", "ollama", "--model", MODEL, "--cell", "classic",
    "--horizon", "52", "--out", "/content/llm_floor_classic.json",
], cwd="/content/beer_distribution_RL")

In [ ]:
#@title 4c) Capability floor on the first-GRPO-cell env (Y-topology, tight capacity)
import subprocess, sys
MODEL = "qwen2.5:3b"
subprocess.check_call([
    sys.executable, "scripts/run_llm_episode.py",
    "--backend", "ollama", "--model", MODEL, "--cell", "y_tight",
    "--horizon", "52", "--out", "/content/llm_floor_y_tight.json",
], cwd="/content/beer_distribution_RL")

In [ ]:
#@title 5) Verdict: did the model play coherently?
import json
for f in ["/content/llm_floor_classic.json", "/content/llm_floor_y_tight.json"]:
    d = json.load(open(f))
    print("===", d["cell"], "| model:", d["model"], "===")
    print("  parse_fail_rate:", d["parse_fail_rate"], "(want ~0)")
    print("  system_total_cost:", d["system_total_cost"])
    print("  mean order per role:", d["mean_order_per_role"])
    print("  first weeks:")
    for w in d["transcript_head"]:
        print("   ", w)
    print()